In [1]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

PROJECT_ROOT = Path.cwd().parent  # nếu notebook nằm trong /notebooks
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

customer_dataset = pd.read_csv(PROCESSED_DIR / "customer_model_dataset.csv")

print(customer_dataset.shape)
customer_dataset.head()
print(customer_dataset.columns.tolist())

(804, 41)
['customer_id', 'gender', 'birth_year', 'city', 'acquisition_channel', 'last_purchase_date', 'first_purchase_date', 'frequency', 'monetary', 'avg_order_value', 'recency_days', 'customer_lifetime_days', 'n_unique_categories', 'n_unique_brands', 'favorite_brand', 'avg_item_price', 'avg_line_amount', 'discount_usage_rate', 'n_transactions', 'support_ticket_count', 'resolved_ticket_count', 'pending_ticket_count', 'avg_satisfaction', 'days_since_last_ticket', 'resolved_rate', 'campaign_count', 'total_emails', 'total_open', 'total_click', 'total_offer_value', 'open_rate', 'click_rate', 'ctr_given_open', 'days_since_last_campaign', 'mean_interpurchase_days', 'std_interpurchase_days', 'last_interpurchase_gap', 'favorite_dow', 'favorite_month', 'will_return_within_30_days', 'days_until_next_purchase']


In [2]:
# CELL 2: Train/test split theo thời gian dùng last_purchase_date

customer_dataset["last_purchase_date"] = pd.to_datetime(
    customer_dataset["last_purchase_date"], errors="coerce"
)

quantile_cutoff = 0.8
time_cutoff = customer_dataset["last_purchase_date"].quantile(quantile_cutoff)

train_mask = customer_dataset["last_purchase_date"] <= time_cutoff
test_mask = customer_dataset["last_purchase_date"] > time_cutoff

train_df = customer_dataset[train_mask].copy()
test_df = customer_dataset[test_mask].copy()

print("Time cutoff:", time_cutoff)
print("Train size:", train_df.shape, "Test size:", test_df.shape)
print("Train positive rate:", train_df["will_return_within_30_days"].mean())
print("Test positive rate:", test_df["will_return_within_30_days"].mean())

Time cutoff: 2026-02-25 04:48:00
Train size: (604, 41) Test size: (151, 41)
Train positive rate: 0.004966887417218543
Test positive rate: 0.006622516556291391


In [3]:
from sklearn.preprocessing import StandardScaler

target_col = "will_return_within_30_days"

drop_cols = [
    "customer_id",
    "will_return_within_30_days",
    "days_until_next_purchase",
    "last_purchase_date",
    "first_purchase_date",
]

feature_cols = [c for c in customer_dataset.columns if c not in drop_cols]

X_train_raw = train_df[feature_cols].copy()
y_train = train_df[target_col]
X_test_raw = test_df[feature_cols].copy()
y_test = test_df[target_col]

# One-hot encode tất cả cột object
X_train = pd.get_dummies(X_train_raw, drop_first=True)
X_test = pd.get_dummies(X_test_raw, drop_first=True)

# Align cột giữa train và test
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

# Impute NaN (ở đây dùng 0 cho đơn giản, hoặc bạn có thể dùng mean/median từng cột)
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print("Any NaN left in X_train?", X_train.isna().any().any())
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

Any NaN left in X_train? False
X_train shape: (604, 74)
X_test shape: (151, 74)


In [4]:
X_train.isna().sum().sort_values(ascending=False).head(10)

birth_year                      0
favorite_brand_Anker            0
acquisition_channel_Website     0
acquisition_channel_Walk-in     0
acquisition_channel_TikTok      0
acquisition_channel_Shopee      0
acquisition_channel_Referral    0
acquisition_channel_Google      0
city_Đà Nẵng                    0
city_Vũng Tàu                   0
dtype: int64

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

# Chuẩn hóa numeric (hầu hết các cột bây giờ là numeric)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
y_proba_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, digits=3))
print("AUC-ROC:", roc_auc_score(y_test, y_proba_lr))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0      0.993     0.973     0.983       150
           1      0.000     0.000     0.000         1

    accuracy                          0.967       151
   macro avg      0.497     0.487     0.492       151
weighted avg      0.987     0.967     0.977       151

AUC-ROC: 0.5966666666666666


In [6]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample"
)

rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf, digits=3))
print("AUC-ROC:", roc_auc_score(y_test, y_proba_rf))

=== Random Forest ===
              precision    recall  f1-score   support

           0      0.993     1.000     0.997       150
           1      0.000     0.000     0.000         1

    accuracy                          0.993       151
   macro avg      0.497     0.500     0.498       151
weighted avg      0.987     0.993     0.990       151

AUC-ROC: 0.9966666666666668


/opt/anaconda3/envs/cs332v-ml/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/cs332v-ml/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/anaconda3/envs/cs332v-ml/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is

In [7]:
feat_imp = (
    pd.DataFrame({
        "feature": X_train.columns,
        "importance": rf.feature_importances_
    })
    .sort_values("importance", ascending=False)
)

feat_imp.head(20)

,feature,importance
28,std_interpurchase_days,0.120636
29,last_interpurchase_gap,0.114028
31,favorite_month,0.065199
0,birth_year,0.054021
7,n_unique_brands,0.053182
30,favorite_dow,0.050759
3,avg_order_value,0.043746
5,customer_lifetime_days,0.043216
41,city_Hải Phòng,0.040972
1,frequency,0.039031
